# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cooper30/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring  
**Audit goal:** Decide what the Week-5 model evidence can honestly support before using it in the public paper.

## 1. Two paper findings + my methodology questions

### Finding 1 — Growing content is longer and younger
The FlyRank March 2026 paper reports an observational comparison: growing pages average about 3.2K words and 184 days of age, while declining pages average about 2.3K words and 230 days. The trend label comes from the 30-day versus previous-30-day impressions comparison.

**Methodology question:** How much of the observed difference remains after accounting for client, content type, and existing visibility? A client-grouped uncertainty estimate or within-client comparison would help show whether the pattern generalizes beyond the largest portfolios. The evidence supports association, not a claim that adding words or being younger causes growth.

### Finding 4 — Recently refreshed mature pages show stronger outcomes
The paper reports that 365+ day content refreshed within 30 days has higher health and impressions than older, less-recently updated content.

**Methodology question:** Were refreshed and untouched pages comparable before the update? Editors may preferentially refresh pages with proven demand, creating selection bias. A matched pre/post design with a comparable untouched group would be needed before interpreting the difference as causal refresh impact. The current result is useful directional evidence for a review program, not proof that refreshing alone produced the lift.

In [1]:
paper_findings = [
    {
        "finding": "Growing content is longer and younger",
        "label_source": "30-day vs previous-30-day impressions trend",
        "safe_claim": "Observed association",
        "validation_question": "Does the pattern remain within clients and content types?",
    },
    {
        "finding": "Recently refreshed mature pages show stronger outcomes",
        "label_source": "Days since update plus observed health and impressions",
        "safe_claim": "Directional comparison, not causal proof",
        "validation_question": "Were refreshed pages matched to comparable untouched pages?",
    },
]

import pandas as pd
display(pd.DataFrame(paper_findings))


,finding,label_source,safe_claim,validation_question
0,Growing content is longer and younger,30-day vs previous-30-day impressions trend,Observed association,Does the pattern remain within clients and con...
1,Recently refreshed mature pages show stronger ...,Days since update plus observed health and imp...,"Directional comparison, not causal proof",Were refreshed pages matched to comparable unt...


## 2. My model under an honest grouped split

Week 5 already used an honest client-grouped 60/20/20 train/validation/test split, so I do not invent a more flattering random-row result. Instead, this audit compares the untouched validation and held-out test behavior and places every ranking metric next to its base rate.

A random ranking has expected Precision@K close to the positive base rate. Therefore, beating the hand-written baseline is not enough if the model still ranks below the base rate.

In [2]:
import json
from pathlib import Path
from urllib.request import urlopen

METRICS_URL = (
    "https://raw.githubusercontent.com/Cooper30/"
    "flyrank-ml-internship/main/work/outputs/w05_model_metrics.json"
)

with urlopen(METRICS_URL) as response:
    metrics = json.load(response)

validation = metrics["validation_candidates"][0]
baseline = metrics["test_baseline"]
model = metrics["test_model"]

audit_table = pd.DataFrame([
    {
        "evaluation": "Validation — selected depth 2",
        "rows": validation["rows"],
        "base_rate": validation["base_rate"],
        "precision_at_20": validation["precision_at_20"],
        "precision_at_50": validation["precision_at_50"],
        "average_precision": validation["average_precision"],
        "roc_auc": validation["roc_auc"],
    },
    {
        "evaluation": "Test — Week-4 rule baseline",
        "rows": baseline["rows"],
        "base_rate": baseline["base_rate"],
        "precision_at_20": baseline["precision_at_20"],
        "precision_at_50": baseline["precision_at_50"],
        "average_precision": baseline["average_precision"],
        "roc_auc": baseline["roc_auc"],
    },
    {
        "evaluation": "Test — Decision Tree depth 2",
        "rows": model["rows"],
        "base_rate": model["base_rate"],
        "precision_at_20": model["precision_at_20"],
        "precision_at_50": model["precision_at_50"],
        "average_precision": model["average_precision"],
        "roc_auc": model["roc_auc"],
    },
])

display(audit_table.round(3))

p50_gain_points = model["precision_at_50"] - baseline["precision_at_50"]
p50_relative_gain = p50_gain_points / baseline["precision_at_50"]
p50_vs_base_rate = model["precision_at_50"] - model["base_rate"]
base_rate_shift = model["base_rate"] - validation["base_rate"]

print(f"P@50 gain over rule baseline: {p50_gain_points:+.3f} ({p50_relative_gain:+.1%})")
print(f"Model P@50 minus test base rate: {p50_vs_base_rate:+.3f}")
print(f"Test minus validation base-rate shift: {base_rate_shift:+.3f}")

assert model["precision_at_50"] > baseline["precision_at_50"]
assert model["precision_at_50"] < model["base_rate"]


,evaluation,rows,base_rate,precision_at_20,precision_at_50,average_precision,roc_auc
0,Validation — selected depth 2,9135,0.339,0.10,0.06,0.327,0.472
1,Test — Week-4 rule baseline,46850,0.558,0.25,0.26,0.525,0.467
2,Test — Decision Tree depth 2,46850,0.558,0.30,0.38,0.583,0.570


P@50 gain over rule baseline: +0.120 (+46.2%)
Model P@50 minus test base rate: -0.178
Test minus validation base-rate shift: +0.218


### Audit verdict

On the grouped test set, the model improves Precision@50 from 0.26 to 0.38, a 0.12-point or approximately 46% relative gain over the rule baseline. However, test base rate is 0.558, so 0.38 remains 0.178 below the expected precision of a random selection from the same test population.

Validation is weaker: Precision@50 is 0.06 against a 0.339 base rate, and ROC-AUC is below 0.50. The 0.218 base-rate shift between validation and test indicates substantial client-level distribution differences. The model therefore shows limited discrimination and unstable cross-client generalization.

## 3. Leakage audit

**Timeline:** March 1–31 features → March 31 decision point → April 1–30 outcome.

- April impressions, decline ratio, and the decline label are not model features.
- Pseudonymous client and content IDs are used only for grouping and joining.
- Client groups do not overlap across train, validation, and test.
- No FlyRank product flags or existing decision scores are model features.
- `observed_days_march` and March CTR account for essentially all tree importance. This is not direct leakage, but the strong reliance on coverage may encode portfolio-specific measurement behavior rather than transferable refresh risk.
- **Population-selection caveat:** clients were required to have at least 20 available GSC days in April. April is the outcome window, so this is future-informed population selection. It does not reveal the page label to the model, but it narrows the evaluated population to clients still sufficiently observed in April and must be disclosed.

In [3]:
importance = pd.DataFrame(metrics["feature_importance"])
display(importance.round(3))

checks = {
    "features_strictly_before_label_window": True,
    "label_or_future_fields_used_as_features": False,
    "ids_used_as_features": False,
    "product_flags_used_as_features": False,
    "client_overlap_across_splits": False,
    "base_rate_reported": True,
    "outcome_window_population_selection_disclosed": True,
    "coverage_signal_requires_monitoring": True,
}

assert metrics["leakage_checks"]["future_fields_used_as_features"] is False
assert metrics["leakage_checks"]["ids_used_as_features"] is False
assert metrics["leakage_checks"]["client_overlap_across_splits"] is False
display(pd.Series(checks, name="audit_result").to_frame())


,feature,importance
0,observed_days_march,0.501
1,ctr_march,0.499
2,log_impressions_march,0.000
3,avg_position_march,0.000
4,content_age_days,0.000


,audit_result
features_strictly_before_label_window,True
label_or_future_fields_used_as_features,False
ids_used_as_features,False
product_flags_used_as_features,False
client_overlap_across_splits,False
base_rate_reported,True
outcome_window_population_selection_disclosed,True
coverage_signal_requires_monitoring,True


## 4. Claim rewrite

**Too strong:**  
The model identifies the pages most likely to decline and should be used to prioritize refreshes.

**Evidence-aligned rewrite:**  
On the client-grouped test set, the depth-2 Decision Tree improved Precision@50 from 0.26 for the hand-written rule to 0.38 and increased ROC-AUC from 0.467 to 0.570. However, its Precision@50 remained below the 0.558 test base rate, validation performance was weak, and the outcome prevalence shifted materially across client groups. The score should therefore be treated as an early decision-support prototype for manual review and further validation, not as an automated refresh recommendation or evidence of causal refresh impact.

In [4]:
audit_receipt = {
    "lane": metrics["lane"],
    "source_metrics": "work/outputs/w05_model_metrics.json",
    "split": metrics["split"],
    "selected_depth": metrics["selected_depth"],
    "validation_base_rate": validation["base_rate"],
    "validation_precision_at_50": validation["precision_at_50"],
    "test_base_rate": model["base_rate"],
    "baseline_precision_at_50": baseline["precision_at_50"],
    "model_precision_at_50": model["precision_at_50"],
    "model_roc_auc": model["roc_auc"],
    "p50_gain_over_baseline_points": p50_gain_points,
    "p50_gap_vs_test_base_rate": p50_vs_base_rate,
    "validation_to_test_base_rate_shift": base_rate_shift,
    "verdict": "LIMITED — beats the hand rule but not the test base rate; unstable across client groups",
    "recommended_use": "manual research and decision support only",
    "leakage_audit": checks,
}

OUTPUT = Path("work/outputs/w06_validation_audit.json")
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(audit_receipt, f, indent=2)

assert OUTPUT.exists()
print("VALIDATION AUDIT PASSED ✅")
print(audit_receipt["verdict"])

from google.colab import files
files.download(str(OUTPUT))


VALIDATION AUDIT PASSED ✅
LIMITED — beats the hand rule but not the test base rate; unstable across client groups


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Self-check

After **Runtime → Run all** completes, confirm:

- [ ] Two FlyRank paper findings are questioned constructively
- [ ] Validation and grouped test results are shown with base rates
- [ ] Model vs baseline uses the same held-out test population
- [ ] Feature/label timeline is explicit
- [ ] Outcome-window population selection is disclosed
- [ ] No future fields, IDs, or product flags are model features
- [ ] The strongest claim is rewritten in decision-support language
- [ ] `VALIDATION AUDIT PASSED ✅` appears
- [ ] `w06_validation_audit.json` is downloaded and committed under `work/outputs/`